# Core Agent Architectures

This file teaches you 9 ways to build an AI agent.

An **agent** is a program where the AI model decides what to do next. It looks at the situation, picks an action, sees the result, and repeats.

By the end, you will know how ReAct, Plan-and-Execute, LATS, Reflexion, CRITIC, Toolformer, Voyager, DEPS, and Huginn agents work.

## Setup

In [1]:
!pip install python_dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

In [8]:
from openai import OpenAI
import json
import time

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

**What happened:** We imported the OpenAI library, `json` for parsing tool calls, and `time` for pauses between API calls.

## 1. ReAct Agent

**What it does:** The agent thinks, uses a tool, reads the result, and repeats until it has an answer. ReAct = Reasoning + Acting.

**When to use it:** When the task needs tools and the agent must reason about which tool to use and what to do with the result.

**Steps:**
1. Think about what to do next.
2. Call a tool (like a calculator or search engine).
3. Read the tool's result.
4. Repeat until done.

### Step 1: Define the tools the agent can use

In [15]:
def calculator(expression):
    result = eval(expression)
    return str(result)

def search_database(query):
    database = {
        "python": "Python is a programming language created by Guido van Rossum in 1991.",
        "javascript": "JavaScript is a programming language created by Brendan Eich in 1995.",
        "rust": "Rust is a systems programming language first released in 2015.",
        "population of france": "The population of France is approximately 68 million.",
        "capital of japan": "The capital of Japan is Tokyo.",
    }
    
    query_lower = query.lower()
    
    for key in database:
        if key in query_lower:
            return database[key]
    return "No information found for: " + query

**What happened:** We created two tools: `calculator` (evaluates math expressions) and `search_database` (looks up facts). In real systems, these would be real APIs.

### Step 2: Tell the model about the tools using OpenAI's function calling format

In [16]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression. Use this for any math question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate, like '25 * 4' or '100 / 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_database",
            "description": "Search for factual information. Use this for any factual question.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    }
                },
                "required": ["query"]
            }
        }
    }
]


for tool in tools:
    name = tool["function"]["name"]
    description = tool["function"]["description"]
    print("  ", name, "-", description)

   calculator - Calculate a math expression. Use this for any math question.
   search_database - Search for factual information. Use this for any factual question.


**What happened:** We described each tool in OpenAI's format with a `name`, `description`, and `parameters`. The model reads these descriptions to decide which tool to call.

### Step 3: Build the ReAct loop

In [20]:
def run_react_agent(question):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Think step by step. Use tools when needed."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    
    for step in range(1, 6):
        print()
        print("--- Step", step, "---")


        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        
        assistant_message = response.choices[0].message
        print(assistant_message)

        
        # Check if the model wants to call a tool
        if assistant_message.tool_calls is None:
            # No tool call — the model has a final answer
            final_answer = assistant_message.content
            print("THINK:", final_answer)
            print()
            print("Agent finished in", step, "steps.")
            return final_answer
        
        
        # The model wants to call one or more tools
        messages.append(assistant_message)

        
        if assistant_message.content:
            print("THINK:", assistant_message.content)

        
        for tool_call in assistant_message.tool_calls:
            
            function_name = tool_call.function.name
            raw_arguments = tool_call.function.arguments
            
            parsed_arguments = json.loads(raw_arguments)


            # Run the tool
            if function_name == "calculator":
                tool_result = calculator(parsed_arguments["expression"])
            elif function_name == "search_database":
                tool_result = search_database(parsed_arguments["query"])
            else:
                tool_result = "Unknown tool: " + function_name

            
            print("OBSERVE:", tool_result)


            # Send the tool result back to the model
            tool_message = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            }
            messages.append(tool_message)
    

    print("Agent did not finish in 5 steps.")
    return None

**What happened:** We built the ReAct loop. It sends the user's question to the model, checks if the model wants a tool, runs the tool if so, sends the result back, and repeats up to 5 steps. This is the Think, Act, Observe loop.

### Step 4: Test the ReAct agent

In [21]:
answer = run_react_agent("What is the population of France divided by 2?")


--- Step 1 ---
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_CY6810rFbx5c3qFsaRPxFq1o', function=Function(arguments='{"query":"current population of France 2023"}', name='search_database'), type='function')])
OBSERVE: The population of France is approximately 68 million.

--- Step 2 ---
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Qc8NJ1mfQi2CJP918F6Hd3DO', function=Function(arguments='{"expression":"68000000 / 2"}', name='calculator'), type='function')])
OBSERVE: 34000000.0

--- Step 3 ---
ChatCompletionMessage(content='The population of France divided by 2 is approximately 34 million.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
THINK: The population of France divided by 2 is

**What happened:** The agent searched for France's population (68 million), then used the calculator to divide by 2 (34 million). It chained two tools to answer one question.

In [19]:
answer = run_react_agent("What year was Python created, and how many years ago was that?")


--- Step 1 ---
OBSERVE: Python is a programming language created by Guido van Rossum in 1991.
OBSERVE: 32

--- Step 2 ---
THINK: Python was created in the year 1991. That was 32 years ago from 2023.

Agent finished in 2 steps.


**What happened:** The agent searched for Python (created in 1991), then used the calculator to find how many years ago that was. It chained search and math tools together.

## 2. Plan-and-Execute Agent

**What it does:** The agent first creates a full plan (a list of steps), then executes each step one at a time. Unlike ReAct, it plans everything before acting.

**When to use it:** When the task has multiple clear steps that can be planned upfront, like "research X, then write Y, then review Z."

**Steps:**
1. Send the task to the model and ask it to make a plan.
2. Execute each step one at a time.
3. Each step sees the results from previous steps.

### Step 1: Ask the model to create a plan

In [22]:
def create_plan(task):
    print("Task:", task)
    print()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a planning agent. "
                    "Break the task into 3-5 simple steps. "
                    "Return ONLY a JSON list of strings. "
                    'Example: ["Step 1: ...", "Step 2: ..."]'
                )
            },
            {
                "role": "user",
                "content": task
            }
        ]
    )

    raw_content = response.choices[0].message.content
    raw_content = raw_content.strip()

    print("RAW CONTENT:\n", raw_content)

    # Remove markdown code fences if present
    if raw_content.startswith("```"):
        lines = raw_content.split("\n")
        lines = lines[1:-1]
        raw_content = "\n".join(lines)

    
    plan = json.loads(raw_content)

    
    print("RAW CONTENT AFTER REMOVAL OF MARKDOWN IF ANY:\n", plan)

    
    print("Plan created with", len(plan), "steps:")
    for i in range(len(plan)):
        print("  ", i + 1, ".", plan[i])

    return plan



task = "Write a short blog post about the benefits of drinking water"
plan = create_plan(task)

Task: Write a short blog post about the benefits of drinking water

RAW CONTENT:
 [
    "Step 1: Research the health benefits of drinking water, including hydration, improved digestion, and skin health.",
    "Step 2: Outline the structure of the blog post, including an introduction, main points, and conclusion.",
    "Step 3: Write the introduction, highlighting why water is essential for life.",
    "Step 4: Expand on each benefit in the main points, providing examples and statistics where applicable.",
    "Step 5: Conclude with a summary of the benefits and a call to action to drink more water."
]
RAW CONTENT AFTER REMOVAL OF MARKDOWN IF ANY:
 ['Step 1: Research the health benefits of drinking water, including hydration, improved digestion, and skin health.', 'Step 2: Outline the structure of the blog post, including an introduction, main points, and conclusion.', 'Step 3: Write the introduction, highlighting why water is essential for life.', 'Step 4: Expand on each benefit in the

**What happened:** The model broke the task into steps and returned a JSON list. This is the Plan phase -- the agent knows all steps before doing any work.

### Step 2: Execute each step one at a time

In [24]:
def execute_step(step_description, previous_results):
    
    context = ""
    
    if len(previous_results) > 0:
        
        context = "Previous results:\n"
        
        for i in range(len(previous_results)):
            context = context + "Step " + str(i + 1) + " result: "
            context = context + previous_results[i][:200] + "\n"

    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are an execution agent. Complete the given step. Be concise."
            },
            {
                "role": "user",
                "content": context + "\nNow execute this step: " + step_description
            }
        ],
        max_tokens=300
    )

    result = response.choices[0].message.content
    return result

**What happened:** We created a function that executes one step at a time. Each step sees all previous results so it can build on them.

In [26]:
def run_plan_and_execute(task):
    plan = create_plan(task)
    print()

    all_results = []
    for i in range(len(plan)):
        step = plan[i]
        print("=" * 50)
        print("Executing step", i + 1, ":", step)
        print()

        result = execute_step(step, all_results)
        all_results.append(result)

        print(result)
        print()

    print("=" * 50)
    print("All", len(plan), "steps completed.")
    return all_results


results = run_plan_and_execute("Explain three benefits of exercise in simple language")

Task: Explain three benefits of exercise in simple language

RAW CONTENT:
 ["Step 1: Identify the physical benefits of exercise, such as improving strength and flexibility.", "Step 2: Highlight the mental health advantages, including reducing stress and anxiety.", "Step 3: Discuss how regular exercise can boost energy levels and improve mood."]
RAW CONTENT AFTER REMOVAL OF MARKDOWN IF ANY:
 ['Step 1: Identify the physical benefits of exercise, such as improving strength and flexibility.', 'Step 2: Highlight the mental health advantages, including reducing stress and anxiety.', 'Step 3: Discuss how regular exercise can boost energy levels and improve mood.']
Plan created with 3 steps:
   1 . Step 1: Identify the physical benefits of exercise, such as improving strength and flexibility.
   2 . Step 2: Highlight the mental health advantages, including reducing stress and anxiety.
   3 . Step 3: Discuss how regular exercise can boost energy levels and improve mood.

Executing step 1 : Step

**What happened:** The agent created a plan, then executed each step in order. The plan was fixed from the start and did not change based on results. This works well when the task structure is predictable.

## 3. LATS (Language Agent Tree Search)

**What it does:** The agent tries multiple approaches, scores each one, and picks the best. LATS = Language Agent Tree Search.

**When to use it:** When the problem has multiple possible solutions and you want the best one. Good for complex reasoning, code generation, or creative tasks.

**Steps:**
1. Generate multiple candidate answers (branches).
2. Score each candidate.
3. Keep the best one.
4. Repeat for more rounds if needed.

In [27]:
def generate_candidates(question, num_candidates):
    
    candidates = []
    for i in range(num_candidates):
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "Answer the question. Be creative and try different approaches."
                },
                {
                    "role": "user",
                    "content": question
                }
            ],
            temperature=0.9,
            max_tokens=200
        )

        answer = response.choices[0].message.content
        candidates.append(answer)

        print("Candidate", i + 1, "generated (" + str(len(answer)), "chars)")

    return candidates


question = "What are three creative ways to reduce plastic waste?"
candidates = generate_candidates(question, 3)

for i in range(len(candidates)):
    print()
    print("--- Candidate", i + 1, "---")
    print(candidates[i], "...")

Candidate 1 generated (1085 chars)
Candidate 2 generated (1202 chars)
Candidate 3 generated (1201 chars)

--- Candidate 1 ---
Certainly! Here are three creative approaches to reducing plastic waste that not only address the issue but also encourage community engagement and innovation:

1. **Plastic-Free Challenge Community Events**:
   Organize local "Plastic-Free Challenge" events where communities compete to go a week or month without using any single-use plastics. Participants can share their strategies and solutions through social media or community boards, creating a supportive environment. To spice it up, include themed days—like “Bulk Buy Day” where participants shop for goods in bulk using reusable containers, or “DIY Day” for creating homemade alternatives to common plastic products (like beeswax wraps instead of cling film). Prizes can be offered for most creative alternatives or for those who manage to use the least plastic.

2. **Eco-Brick Workshops**:
   Host workshops whe

**What happened:** We generated 3 different answers using high temperature (0.9) to make each candidate different. These are the "branches" of the tree.

In [28]:
def score_candidate(question, candidate):
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a judge. Score the answer from 1 to 10. "
                    "Consider: accuracy, creativity, completeness, clarity. "
                    "Reply with ONLY a number."
                )
            },
            {
                "role": "user",
                "content": "Question: " + question + "\n\nAnswer: " + candidate
            }
        ],
        max_tokens=5
    )

    raw_score = response.choices[0].message.content
    raw_score = raw_score.strip()

    # Extract the number
    digits = ""
    for char in raw_score:
        if char.isdigit():
            digits = digits + char
    if digits == "":
        return 5
    score = int(digits)
    return score


scores = []
for i in range(len(candidates)):
    score = score_candidate(question, candidates[i])
    scores.append(score)
    print("Candidate", i + 1, "score:", score)

Candidate 1 score: 5
Candidate 2 score: 8
Candidate 3 score: 8


**What happened:** We asked the model to judge each candidate on a 1-10 scale. LATS uses these scores to focus on the most promising paths.

In [29]:
def run_lats(question, num_candidates=3, num_rounds=2):
    print("Question:", question)
    print()

    best_answer = ""
    best_score = 0

    for round_num in range(1, num_rounds + 1):
        print("=== Round", round_num, "===")

        candidates = generate_candidates(question, num_candidates)

        for i in range(len(candidates)):
            score = score_candidate(question, candidates[i])
            print("  Candidate", i + 1, "score:", score)

            if score > best_score:
                best_score = score
                best_answer = candidates[i]
                print("  ^ New best!")

        print()

    print("Best score:", best_score)
    print("Best answer:", best_answer[:300])
    return best_answer


best = run_lats("Explain quantum computing to a 10 year old", num_candidates=3, num_rounds=2)

Question: Explain quantum computing to a 10 year old

=== Round 1 ===
Candidate 1 generated (945 chars)
Candidate 2 generated (922 chars)
Candidate 3 generated (847 chars)
  Candidate 1 score: 9
  ^ New best!
  Candidate 2 score: 8
  Candidate 3 score: 9

=== Round 2 ===
Candidate 1 generated (877 chars)
Candidate 2 generated (895 chars)
Candidate 3 generated (871 chars)
  Candidate 1 score: 9
  Candidate 2 score: 8
  Candidate 3 score: 8

Best score: 9
Best answer: Okay, imagine you have a super special magic box called a quantum computer. This box is different from your regular computer, which is like a really fast calculator that helps you do math, play games, and find stuff online.

Now, let’s pretend that regular computers think in "light switches" – they 


**What happened:** LATS ran 2 rounds, generating 3 candidates per round (6 total). It scored each and kept the highest-scoring answer. In a full implementation, later rounds would build on the best candidates from earlier rounds.

## 4. Reflexion

**What it does:** The agent tries to solve a problem, reflects on mistakes, stores lessons in memory, and retries. It improves through self-reflection.

**When to use it:** When the agent might make mistakes and you want it to learn from its own errors. Good for code generation and reasoning.

**Steps:**
1. Attempt the task.
2. Evaluate the answer.
3. If wrong, reflect on what went wrong and store the lesson.
4. Retry with past reflections in memory.

In [31]:
def attempt_task(task, past_reflections):
    
    reflection_text = ""
    
    if len(past_reflections) > 0:
        reflection_text = "\nYour past mistakes and lessons:\n"
        for i in range(len(past_reflections)):
            reflection_text = reflection_text + "- " + past_reflections[i] + "\n"
        reflection_text = reflection_text + "\nAvoid these mistakes this time.\n"

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a careful problem solver." + reflection_text
            },
            {
                "role": "user",
                "content": task
            }
        ],
        max_tokens=300
    )

    answer = response.choices[0].message.content
    return answer

**What happened:** We created a function that attempts a task. Past reflections (lessons from failures) are included in the prompt so the model avoids previous mistakes.

In [32]:
def evaluate_answer(task, answer):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict evaluator. "
                    "Check if the answer is correct and complete. "
                    'Reply with JSON: {"correct": true/false, "issues": "description of problems"}'
                )
            },
            {
                "role": "user",
                "content": "Task: " + task + "\n\nAnswer: " + answer
            }
        ],
        max_tokens=200
    )

    raw = response.choices[0].message.content
    raw = raw.strip()
    if raw.startswith("```"):
        lines = raw.split("\n")
        lines = lines[1:-1]
        raw = "\n".join(lines)

    try:
        evaluation = json.loads(raw)
    except:
        evaluation = {"correct": False, "issues": "Could not parse evaluation"}

    return evaluation

**What happened:** We created an evaluator that returns `correct` (true/false) and `issues` (what went wrong). The agent uses this to judge its own work.

In [33]:
def reflect_on_failure(task, answer, issues):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are reflecting on a failed attempt. "
                    "Write ONE sentence: what went wrong and what to do differently next time."
                )
            },
            {
                "role": "user",
                "content": (
                    "Task: " + task +
                    "\nMy answer: " + answer[:200] +
                    "\nIssues found: " + issues
                )
            }
        ],
        max_tokens=100
    )

    reflection = response.choices[0].message.content
    return reflection

**What happened:** We created a reflection function that writes a one-sentence lesson when the agent fails. This lesson is stored and used in the next attempt.

In [34]:
def run_reflexion_agent(task, max_attempts=3):
    print("Task:", task)
    print()

    past_reflections = []

    for attempt_num in range(1, max_attempts + 1):
        print("--- Attempt", attempt_num, "---")

        answer = attempt_task(task, past_reflections)
        print("Answer:", answer[:200])
        print()

        evaluation = evaluate_answer(task, answer)
        is_correct = evaluation.get("correct", False)
        issues = evaluation.get("issues", "none")

        print("Correct:", is_correct)
        print("Issues:", issues)
        print()

        if is_correct:
            print("Success on attempt", attempt_num)
            return answer

        reflection = reflect_on_failure(task, answer, issues)
        past_reflections.append(reflection)
        print("Reflection:", reflection)
        print()

    print("Failed after", max_attempts, "attempts.")
    return answer


result = run_reflexion_agent(
    "List exactly 5 European countries that start with the letter 'S', with their capitals."
)

Task: List exactly 5 European countries that start with the letter 'S', with their capitals.

--- Attempt 1 ---
Answer: Here are five European countries that start with the letter 'S' along with their capitals:

1. Spain - Madrid
2. Sweden - Stockholm
3. Switzerland - Bern
4. Serbia - Belgrade
5. Slovakia - Bratislava

Correct: True
Issues: 

Success on attempt 1


**What happened:** The Reflexion agent attempted the task, evaluated itself, reflected on mistakes, and retried with lessons in memory. Each attempt gets better because the agent remembers past mistakes.

## 5. CRITIC

**What it does:** The agent generates an answer, then uses tools to verify each claim and correct errors. CRITIC = Correcting with tool-interactive critiquing.

**When to use it:** When factual accuracy matters and you have verification tools. Unlike Reflexion (which reflects using reasoning alone), CRITIC uses external tools to verify facts.

**Steps:**
1. Generate an initial answer.
2. Verify claims using tools (search, calculator, etc.).
3. Correct any errors found.
4. Verify again if needed.

In [35]:
def generate_initial_answer(question):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=300
    )
    answer = response.choices[0].message.content
    return answer


def verify_with_tools(answer, question):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a fact-checker. "
                    "Check the answer for factual errors. "
                    "For each claim, say if it is CORRECT or WRONG. "
                    'Reply with JSON: {"has_errors": true/false, "errors": ["error 1", "error 2"], "corrections": ["correction 1", "correction 2"]}'
                )
            },
            {
                "role": "user",
                "content": "Question: " + question + "\n\nAnswer to verify: " + answer
            }
        ],
        max_tokens=300
    )
    
    
    raw = response.choices[0].message.content
    raw = raw.strip()
    
    if raw.startswith("```"):
        lines = raw.split("\n")
        lines = lines[1:-1]
        raw = "\n".join(lines)
    
    
    try:
        result = json.loads(raw)
    except:
        result = {"has_errors": False, "errors": [], "corrections": []}
    return result

**What happened:** We created `generate_initial_answer` (gives a first answer) and `verify_with_tools` (checks for factual errors). In a real system, the verifier would use actual search or database tools.

In [36]:
def correct_answer(original_answer, errors, corrections):
    
    correction_text = ""
    for i in range(len(errors)):
        correction_text = correction_text + "- Error: " + errors[i]
        if i < len(corrections):
            correction_text = correction_text + " -> Fix: " + corrections[i]
        correction_text = correction_text + "\n"

    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Rewrite the answer with the corrections applied. Keep the same structure."
            },
            {
                "role": "user",
                "content": (
                    "Original answer:\n" + original_answer +
                    "\n\nCorrections to apply:\n" + correction_text
                )
            }
        ],
        max_tokens=300
    )
    corrected = response.choices[0].message.content
    return corrected




def run_critic_agent(question, max_rounds=2):
    print("Question:", question)
    print()

    answer = generate_initial_answer(question)
    print("Initial answer:", answer[:300])
    print()

    for round_num in range(1, max_rounds + 1):
        print("--- Verification round", round_num, "---")

        verification = verify_with_tools(answer, question)
        has_errors = verification.get("has_errors", False)
        errors = verification.get("errors", [])
        corrections = verification.get("corrections", [])

        print("Has errors:", has_errors)
        if has_errors:
            print("Errors found:", errors)
            print("Corrections:", corrections)
            answer = correct_answer(answer, errors, corrections)
            print()
            print("Corrected answer:", answer[:300])
        else:
            print("Answer verified as correct.")
            break
        print()

    return answer


result = run_critic_agent("What is the capital of Australia and what is its population?")

Question: What is the capital of Australia and what is its population?

Initial answer: The capital of Australia is Canberra. As of the latest estimates, which may vary, the population of Canberra is around 450,000 people. For the most up-to-date figures, it's advisable to check the latest census data or demographic reports.

--- Verification round 1 ---
Has errors: False
Answer verified as correct.


**What happened:** The CRITIC agent generated an answer, verified it with a fact-checker, corrected errors, and verified again. The key idea: verify with tools, then fix errors.

## 6. Toolformer

**What it does:** The model decides on its own when and where to use a tool. It inserts tool calls (like `[CALC: 150 * 24]`) into its response wherever it thinks a tool would help.

**When to use it:** When you want the model to autonomously decide about tool use, rather than always or never using tools.

**Steps:**
1. The model writes a response with tool markers where needed.
2. We find and execute each marker.
3. We replace markers with real results.

In [39]:
def toolformer_respond(question):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an assistant that decides when to use tools. "
                    "When you need to calculate something, write [CALC: expression]. "
                    "When you need to look something up, write [SEARCH: query]. "
                    "Write your response with these markers where needed."
                )
            },
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=300
    )

    raw_response = response.choices[0].message.content

    print("RAW RESPONSE:", raw_response + "\nEND\n")

    return raw_response


raw = toolformer_respond("If I save $150 per month for 2 years, how much will I have?")
print("Raw response with tool markers:")
print(raw)

RAW RESPONSE: To find out how much you'll have after saving $150 per month for 2 years, I can calculate it as follows:

Total savings = monthly savings × number of months

You save $150 per month for 2 years, which is 2 × 12 = 24 months.

So, the calculation would be:
Total savings = $150 × 24

Now I will calculate that. 

[CALC: 150 * 24]
END

Raw response with tool markers:
To find out how much you'll have after saving $150 per month for 2 years, I can calculate it as follows:

Total savings = monthly savings × number of months

You save $150 per month for 2 years, which is 2 × 12 = 24 months.

So, the calculation would be:
Total savings = $150 × 24

Now I will calculate that. 

[CALC: 150 * 24]


**What happened:** The model inserted tool markers like `[CALC: 150 * 24]` where it decided a calculation was needed. It chose on its own when to use a tool.

In [44]:
import re

def execute_tool_markers(text):
    print("Processing tool markers...")
    print()

    # Find all [CALC: ...] markers
    calc_pattern = r'\[CALC:\s*([^\]]+)\]'
    calc_matches = re.findall(calc_pattern, text)
    print("calc_matches\n", calc_matches, "\n")
    
    
    for match in calc_matches:
        
        expression = match.strip()
        
        try:
            result = str(eval(expression))
        except:
            result = "ERROR"
        
        old_marker = "[CALC: " + match + "]"
        
        text = text.replace(old_marker, result)

        print("TEXT START\n", text, "\nTEXT END\n")
        
        print("  CALC:", expression, "=", result)

    
    
    # Find all [SEARCH: ...] markers
    search_pattern = r'\[SEARCH:\s*([^\]]+)\]'
    search_matches = re.findall(search_pattern, text)
    
    for match in search_matches:
        query = match.strip()
        search_result = search_database(query)
        
        old_marker = "[SEARCH: " + match + "]"
        text = text.replace(old_marker, search_result)
        
        print("  SEARCH:", query, "->", search_result[:80])

    return text


final_response = execute_tool_markers(raw)
print()
print("Final response:")
print(final_response)

Processing tool markers...

calc_matches
 ['150 * 24'] 

TEXT START
 To find out how much you'll have after saving $150 per month for 2 years, I can calculate it as follows:

Total savings = monthly savings × number of months

You save $150 per month for 2 years, which is 2 × 12 = 24 months.

So, the calculation would be:
Total savings = $150 × 24

Now I will calculate that. 

3600 
TEXT END

  CALC: 150 * 24 = 3600

Final response:
To find out how much you'll have after saving $150 per month for 2 years, I can calculate it as follows:

Total savings = monthly savings × number of months

You save $150 per month for 2 years, which is 2 × 12 = 24 months.

So, the calculation would be:
Total savings = $150 × 24

Now I will calculate that. 

3600


**What happened:** We found tool markers in the response, executed each one (ran calculations or searches), and replaced them with real results. The final response reads naturally.

## 7. Voyager

**What it does:** As the agent solves tasks, it saves successful solutions as reusable "skills." Next time it sees a similar task, it reuses the skill instead of solving from scratch. Voyager = open-ended agent with a skill library.

**When to use it:** For long-running agents that encounter similar tasks repeatedly. The agent gets faster over time.

**Steps:**
1. Receive a new task.
2. Check the skill library for a matching skill.
3. If found, reuse it. If not, solve it and save the solution as a new skill.

In [47]:
skill_library = {}

def save_skill(task_type, solution):
    skill_library[task_type] = solution
    print("Saved skill:", task_type)


def find_skill(task):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You classify tasks into categories. "
                    "Available categories: " + str(list(skill_library.keys())) + ". "
                    "If the task matches a category, reply with ONLY the category name. "
                    "If no match, reply with NONE."
                )
            },
            {
                "role": "user",
                "content": task
            }
        ],
        max_tokens=50
    )
    
    category = response.choices[0].message.content
    category = category.strip()

    if category in skill_library:
        return category
    return None

print("Skill library is empty. Agent will learn as it works.")

Skill library is empty. Agent will learn as it works.


**What happened:** We created a skill library (dictionary) with `save_skill` (stores a solution) and `find_skill` (checks if a new task matches an existing skill). The library starts empty.

In [48]:
def solve_task(task):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Solve this task. Then write a REUSABLE TEMPLATE for similar tasks. "
                    "Format:\n"
                    "SOLUTION: <your answer>\n"
                    "TEMPLATE: <reusable pattern for similar tasks>"
                )
            },
            {
                "role": "user",
                "content": task
            }
        ],
        max_tokens=400
    )
    
    result = response.choices[0].message.content
    return result


def run_voyager_agent(task):
    
    print("Task:", task)

    # Check skill library
    matching_skill = find_skill(task)

    if matching_skill is not None:
        print("Found existing skill:", matching_skill)
        print("Using saved template:")
        print(skill_library[matching_skill][:200])
        print()
        return skill_library[matching_skill]

    # No existing skill — solve from scratch
    print("No matching skill found. Solving from scratch...")
    result = solve_task(task)
    print(result[:300])
    print()

    # Save as new skill
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "What category does this task belong to? Reply with 2-3 words."
            },
            {
                "role": "user",
                "content": task
            }
        ],
        max_tokens=20
    )
    category = response.choices[0].message.content
    category = category.strip().lower()
    save_skill(category, result)

    return result

**What happened:** We built the Voyager loop: check for a matching skill, reuse it if found, otherwise solve from scratch and save the solution. The agent never solves the same type of problem twice.

In [49]:
print("=== First task (no skills yet) ===")
run_voyager_agent("Write a professional email declining a meeting invitation")
print()

print("=== Second task (should match existing skill) ===")
run_voyager_agent("Write a professional email asking to reschedule a meeting")
print()

print("Skill library now has", len(skill_library), "skills:")
for name in skill_library:
    print("  -", name)

=== First task (no skills yet) ===
Task: Write a professional email declining a meeting invitation
No matching skill found. Solving from scratch...
SOLUTION: 

Subject: Meeting Invitation

Dear [Recipient's Name],

Thank you for your invitation to the meeting scheduled for [Date and Time]. I appreciate the opportunity to connect and discuss [brief mention of the meeting topic].

Unfortunately, I must decline due to prior commitments. I value th

Saved skill: professional communication

=== Second task (should match existing skill) ===
Task: Write a professional email asking to reschedule a meeting
Found existing skill: professional communication
Using saved template:
SOLUTION: 

Subject: Meeting Invitation

Dear [Recipient's Name],

Thank you for your invitation to the meeting scheduled for [Date and Time]. I appreciate the opportunity to connect and discuss [brie


Skill library now has 1 skills:
  - professional communication


## 8. DEPS (Describe, Explain, Plan, Select)

**What it does:** A 4-step problem-solving approach: Describe the situation, Explain the goal, generate multiple Plans, then Select and execute the best one.

**When to use it:** For complex tasks where you need to understand the situation before acting. Good for ambiguous tasks with multiple approaches.

**Steps:**
1. Describe the current situation.
2. Explain the goal and what success looks like.
3. Generate multiple possible plans.
4. Select the best plan and execute it.

In [50]:
def run_deps_agent(task):
    print("Task:", task)
    print()

    # Step 1: DESCRIBE
    print("--- Step 1: DESCRIBE ---")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Describe the current situation and context for this task. Be factual. 2-3 sentences."
            },
            {"role": "user", "content": task}
        ],
        max_tokens=150
    )
    description = response.choices[0].message.content
    print(description)
    print()

    # Step 2: EXPLAIN
    print("--- Step 2: EXPLAIN ---")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Explain what the goal is and why it matters. What does success look like? 2-3 sentences."
            },
            {"role": "user", "content": "Situation: " + description + "\nTask: " + task}
        ],
        max_tokens=150
    )
    explanation = response.choices[0].message.content
    print(explanation)
    print()

    # Step 3: PLAN
    print("--- Step 3: PLAN (generate 3 options) ---")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Generate exactly 3 different plans to accomplish the goal. "
                    "Format each as: Plan 1: ... Plan 2: ... Plan 3: ..."
                )
            },
            {
                "role": "user",
                "content": (
                    "Situation: " + description +
                    "\nGoal: " + explanation +
                    "\nTask: " + task
                )
            }
        ],
        max_tokens=400
    )
    plans = response.choices[0].message.content
    print(plans)
    print()

    # Step 4: SELECT
    print("--- Step 4: SELECT ---")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a decision maker. Pick the best plan and explain why in 1-2 sentences. "
                    "Then execute it by providing the final output."
                )
            },
            {
                "role": "user",
                "content": (
                    "Task: " + task +
                    "\nPlans:\n" + plans
                )
            }
        ],
        max_tokens=300
    )
    selection = response.choices[0].message.content
    print(selection)

    return selection


result = run_deps_agent("Create a study schedule for someone learning Python in 4 weeks")

Task: Create a study schedule for someone learning Python in 4 weeks

--- Step 1: DESCRIBE ---
In the current context, many individuals are seeking to enhance their programming skills, particularly in Python, which is known for its versatility and ease of use. With an increasing demand for Python developers in various industries, learners are motivated to effectively structure their study time, particularly over a period of four weeks. The following study schedule provides a focused plan for mastering the fundamentals and some intermediate concepts of Python within this timeframe. 

### Week 1: Python Basics
- **Day 1**: Introduction to Python and installation (Anaconda or Python.org), setting up IDE (e.g., PyCharm, VSCode)
- **Day 2**: Basic syntax, data types (int, float, string, boolean), and operators
- **Day 

--- Step 2: EXPLAIN ---
The goal of this study schedule is to provide a structured approach for individuals aiming to enhance their Python programming skills within a four-w

**What happened:** The DEPS agent described the situation, explained the goal, generated 3 plans, and selected the best one. Each phase built on the previous one, giving the agent deep understanding before acting.

## 9. Huginn (Hierarchical Agent)

**What it does:** A top-level agent breaks a complex task into independent sub-goals. Each sub-goal is solved by a separate agent. Results flow back up for the final answer. Huginn = hierarchical sub-goal decomposition.

**When to use it:** For complex tasks with multiple independent parts that can be solved separately and combined.

**Steps:**
1. Top-level agent breaks the task into sub-goals.
2. Each sub-goal agent solves its piece.
3. Top-level agent combines all results into a final answer.

In [52]:
def decompose_into_subgoals(task):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Break this task into 2-4 independent sub-goals. "
                    "Each sub-goal should be solvable on its own. "
                    "Return a JSON list of strings."
                )
            },
            {"role": "user", "content": task}
        ],
        max_tokens=300
    )

    raw = response.choices[0].message.content
    raw = raw.strip()

    if raw.startswith("```"):
        lines = raw.split("\n")
        lines = lines[1:-1]
        raw = "\n".join(lines)
        
    subgoals = json.loads(raw)
    return subgoals


def solve_subgoal(subgoal):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a specialist. Solve this sub-goal thoroughly. Be concise."
            },
            {"role": "user", "content": subgoal}
        ],
        max_tokens=200
    )
    result = response.choices[0].message.content
    return result


def combine_results(task, subgoal_results):
    results_text = ""
    for i in range(len(subgoal_results)):
        results_text = results_text + "\nPart " + str(i + 1) + ": " + subgoal_results[i]

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Combine these partial results into one coherent final answer."
            },
            {
                "role": "user",
                "content": "Original task: " + task + "\n\nPartial results:" + results_text
            }
        ],
        max_tokens=400
    )
    final = response.choices[0].message.content
    return final

**What happened:** We created `decompose_into_subgoals` (breaks the task apart), `solve_subgoal` (solves one piece), and `combine_results` (merges everything into a final answer).

In [53]:
def run_huginn_agent(task):
    print("Task:", task)
    print()

    # Step 1: Decompose
    print("--- Decomposing into sub-goals ---")
    subgoals = decompose_into_subgoals(task)
    for i in range(len(subgoals)):
        print("  Sub-goal", i + 1, ":", subgoals[i])
    print()

    # Step 2: Solve each sub-goal
    subgoal_results = []
    for i in range(len(subgoals)):
        print("--- Solving sub-goal", i + 1, "---")
        result = solve_subgoal(subgoals[i])
        subgoal_results.append(result)
        print(result[:200])
        print()

    # Step 3: Combine
    print("--- Combining results ---")
    final_answer = combine_results(task, subgoal_results)
    print(final_answer[:400])

    return final_answer


result = run_huginn_agent("Compare Python and JavaScript: syntax, use cases, and job market")

Task: Compare Python and JavaScript: syntax, use cases, and job market

--- Decomposing into sub-goals ---
  Sub-goal 1 : Compare the syntax of Python and JavaScript, highlighting key differences and similarities.
  Sub-goal 2 : Identify and discuss common use cases for Python and JavaScript in various fields such as web development, data science, and application development.
  Sub-goal 3 : Analyze the current job market for Python and JavaScript developers, including demand, salaries, and job growth trends.
  Sub-goal 4 : Provide examples of popular frameworks or libraries associated with Python and JavaScript and their specific applications.

--- Solving sub-goal 1 ---
Python and JavaScript are both high-level programming languages, but they have distinct syntax and features. Here are the key differences and similarities:

### Similarities:
1. **High-level Construct

--- Solving sub-goal 2 ---
### Common Use Cases for Python

1. **Web Development**:
   - **Frameworks**: Python is com

**What happened:** The agent broke the task into independent sub-goals, solved each one separately with a specialist, and combined all results into one final answer. This is hierarchical decomposition -- delegate work, then synthesize.

## Summary

| Agent | Key Idea | Loop |
|---|---|---|
| **ReAct** | Think → Act → Observe | Reason, use tool, observe result, repeat |
| **Plan-and-Execute** | Plan all steps, then execute | Make plan upfront, follow it step by step |
| **LATS** | Tree search over multiple paths | Generate candidates, score, keep best |
| **Reflexion** | Learn from mistakes | Attempt → evaluate → reflect → retry |
| **CRITIC** | Verify with tools | Generate → verify facts → correct errors |
| **Toolformer** | Model decides when to use tools | Generate text with tool markers, fill them in |
| **Voyager** | Build a skill library | Solve task, save skill, reuse next time |
| **DEPS** | Describe → Explain → Plan → Select | Understand first, then pick best approach |
| **Huginn** | Hierarchical decomposition | Break into sub-goals, solve each, combine |

**When to pick which:**
- Simple tool use → **ReAct**
- Predictable multi-step tasks → **Plan-and-Execute**
- Need the best answer from many options → **LATS**
- Want self-improvement over attempts → **Reflexion**
- Factual accuracy is critical → **CRITIC**
- Model should autonomously decide about tools → **Toolformer**
- Long-running agent with repeated tasks → **Voyager**
- Complex ambiguous tasks → **DEPS**
- Large tasks with independent parts → **Huginn**